In [7]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import os
import math
from collections import defaultdict, Counter
import pickle
import heapq

In [8]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(row):
    text = f"{row['case_id']} {row['case_title']} {row['case_outcome']} {row['case_text']}".lower()
    text = re.sub(r'[^a-z\s]', '', text)  # Eliminar signos
    tokens = text.split()
    tokens = [stemmer.stem(t) for t in tokens if t not in stop_words]
    return tokens

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\davie\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
def build_spimi_blocks(df, block_size=10000, output_dir="index_blocks"):
    os.makedirs(output_dir, exist_ok=True)
    norms = {}
    df_counter = Counter()
    block_count = 0

    for start in range(0, len(df), block_size):
        end = min(start + block_size, len(df))
        block_df = df.iloc[start:end]
        block_index = defaultdict(list)

        for _, row in block_df.iterrows():
            doc_id = row["case_id"]
            tokens = preprocess_text(row)
            tf = Counter(tokens)
            doc_len = 0

            for term, freq in tf.items():
                tf_weight = 1 + math.log10(freq)
                block_index[term].append((doc_id, tf_weight))
                df_counter[term] += 1
                doc_len += tf_weight ** 2

            norms[doc_id] = math.sqrt(doc_len)

        with open(f"{output_dir}/block_{block_count}.pkl", "wb") as f:
            pickle.dump(dict(block_index), f)

        block_count += 1

    # Guardar normas y df_counter
    with open(f"{output_dir}/norms.pkl", "wb") as f:
        pickle.dump(norms, f)
    with open(f"{output_dir}/df_counter.pkl", "wb") as f:
        pickle.dump(df_counter, f)

    return block_count

In [10]:
def query_from_blocks(query, index_dir="index_blocks", k=5):
    with open(f"{index_dir}/norms.pkl", "rb") as f:
        norms = pickle.load(f)
    with open(f"{index_dir}/df_counter.pkl", "rb") as f:
        df_counter = pickle.load(f)

    query_tokens = preprocess_text({'case_id': '', 'case_title': '', 'case_outcome': '', 'case_text': query})
    tf = Counter(query_tokens)
    scores = defaultdict(float)
    query_len = 0
    total_docs = len(norms)

    # Obtener bloques
    block_files = [f for f in os.listdir(index_dir) if f.startswith("block_") and f.endswith(".pkl")]
    block_files.sort()  # Ordenar

    for term, freq in tf.items():
        tf_weight = 1 + math.log10(freq)
        df = df_counter.get(term, 1)
        idf = math.log10(total_docs / df)
        w_tq = tf_weight * idf
        query_len += w_tq ** 2

        for file in block_files:
            with open(os.path.join(index_dir, file), "rb") as f:
                block_index = pickle.load(f)
                if term in block_index:
                    for doc_id, w_td in block_index[term]:
                        scores[doc_id] += w_td * w_tq

    query_norm = math.sqrt(query_len)
    for doc_id in scores:
        scores[doc_id] /= (norms[doc_id] * query_norm)

    return heapq.nlargest(k, scores.items(), key=lambda x: x[1])

In [11]:
# Leer CSV
df = pd.read_csv("../data/legal_text_classification.csv")

block_count = build_spimi_blocks(df, block_size=10000)

In [12]:
results = query_from_blocks("The general principles governing the exercise of the discretion to award indemnity costs")
for doc_id, score in results:
    print(f"Caso {doc_id[4:]} → Score: {score:.4f}")

Caso 16456 → Score: 0.3927
Caso 2 → Score: 0.3767
Caso 4 → Score: 0.3702
Caso 23068 → Score: 0.3319
Caso 5909 → Score: 0.3272
